# Task 1 — Thematic Factor Discovery: Reasoning Model Analysis

Combines returns generation with statistical + LLM reasoning to identify which extracted factors
connect to post-filing stock performance.

**Part A — Data Preparation (Steps 1–4, no GPU)**
1. Load scored factor JSONs
2. Generate 21-day post-filing returns via `yfinance`
3. Merge factors with returns

**Part B — Analysis & Reasoning (Steps 5–9)**
4. Factor coverage analysis
5. Sentiment visualizations
6. Factor evolution & statistical analysis
7. LLM pairwise + synthesis reasoning (requires vLLM on ACCRE)
8. Display reasoning results

| | |
|---|---|
| **Input** | `output/factors_scored/{TICKER}/*_factors.json` |
| **Input** | AAL stock prices via `yfinance` (downloaded at runtime) |
| **Output** | `output/filing_returns.csv` — 21-day post-filing returns |
| **Output** | `output/reasoning_output.json` — LLM reasoning results |

> Steps 1–7 run locally (no GPU). Steps 8–9 require vLLM on ACCRE.

## Step 1: Imports & Configuration

In [ ]:
import json
import logging
import sys
import time
import threading
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import yfinance as yf
except ImportError:
    raise ImportError(
        "yfinance is required for returns generation. "
        "Install it with: pip install yfinance"
    )

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None  # only needed for Steps 8-9

from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()

_search = _cwd
FILLINGS_ROOT = None
for _ in range(6):
    if (_search / "Data").is_dir():
        FILLINGS_ROOT = _search
        break
    _search = _search.parent

if FILLINGS_ROOT is None:
    raise FileNotFoundError(
        f"Could not find Data/ directory in any parent of {_cwd}. "
        "Make sure the notebook is inside the fillings/ project tree."
    )

FACTORS_SCORED_DIR = _cwd / "output" / "factors_scored"
RETURNS_CSV_PATH = _cwd / "output" / "filing_returns.csv"
REASONING_OUTPUT_PATH = _cwd / "output" / "reasoning_output.json"
TICKER_MAPPING_PATH = _cwd / "ticker_mapping.json"

# ── LLM Constants ────────────────────────────────────────────────────────
BASE_URL = "http://127.0.0.1:8000/v1"
API_KEY = "local"
MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
TEMPERATURE = 0.0
MAX_RETRIES = 5
RETRY_BASE_DELAY = 2.0
REQUEST_TIMEOUT = 600  # reasoning models are slower
MAX_IN_FLIGHT_LLM = 4

# ── Sentiment label → numeric score ──────────────────────────────────────
LABEL_TO_SCORE = {
    "very_negative": -2,
    "negative": -1,
    "neutral": 0,
    "positive": 1,
    "very_positive": 2,
}

# ── Ticker filter (comment out to run all tickers) ───────────────────────
TICKERS = ["AAL"]

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_1_5")

print(f"Fillings root:          {FILLINGS_ROOT}")
print(f"Scored factors dir:     {FACTORS_SCORED_DIR} (exists={FACTORS_SCORED_DIR.exists()})")
print(f"Returns CSV:            {RETURNS_CSV_PATH}")
print(f"Reasoning output:       {REASONING_OUTPUT_PATH}")
print(f"Ticker mapping:         {TICKER_MAPPING_PATH} (exists={TICKER_MAPPING_PATH.exists()})")
print(f"Tickers:                {TICKERS}")
print(f"Model:                  {MODEL}")

## Step 2: Load Scored Factor Data

Walk `output/factors_scored/{TICKER}/` and build a flat DataFrame of all factors with their sentiment labels.

In [ ]:
rows = []
for ticker in TICKERS:
    ticker_dir = FACTORS_SCORED_DIR / ticker
    if not ticker_dir.exists():
        log.warning(f"No scored factors for {ticker}")
        continue
    for fpath in sorted(ticker_dir.glob("*_factors.json")):
        with open(fpath) as f:
            data = json.load(f)
        filing_ticker = data["ticker"]
        form = data["form"]
        filing_date = data["filing_date"]
        for fac in data.get("factors", []):
            sentiment = fac.get("sentiment")
            if not isinstance(sentiment, dict) or not sentiment.get("label"):
                continue
            rows.append({
                "ticker": filing_ticker,
                "form": form,
                "filing_date": filing_date,
                "key": fac["key"],
                "category": fac["category"],
                "summary": fac["summary"],
                "label": sentiment["label"],
                "rationale": sentiment.get("rationale", ""),
                "confidence": sentiment.get("confidence", 0.5),
            })

factors_df = pd.DataFrame(rows)
factors_df["filing_date"] = pd.to_datetime(factors_df["filing_date"])
factors_df["score"] = factors_df["label"].map(LABEL_TO_SCORE)
factors_df = factors_df.sort_values(["ticker", "filing_date", "key"]).reset_index(drop=True)

print(f"Shape: {factors_df.shape}")
print(f"Unique tickers: {factors_df['ticker'].nunique()}")
print(f"Unique filings: {factors_df.groupby(['ticker', 'filing_date']).ngroups}")
print(f"Unique factor keys: {factors_df['key'].nunique()}")
print(f"\nLabel distribution:")
label_order = ["very_negative", "negative", "neutral", "positive", "very_positive"]
print(factors_df["label"].value_counts().reindex(label_order).to_string())
factors_df.head(10)

## Step 3: Generate Filing Returns

Download prices via `yfinance`, compute 21-trading-day returns and excess returns vs SPY.

In [ ]:
# Unique (ticker, filing_date) pairs from factor data
filing_pairs = (
    factors_df[["ticker", "form", "filing_date"]]
    .drop_duplicates()
    .sort_values("filing_date")
    .reset_index(drop=True)
)

tickers_list = filing_pairs["ticker"].unique().tolist()
download_tickers = list(set(tickers_list + ["SPY"]))

min_date = filing_pairs["filing_date"].min() - timedelta(days=30)
max_date = filing_pairs["filing_date"].max() + timedelta(days=60)

print(f"Downloading prices for {download_tickers} from {min_date.date()} to {max_date.date()}...")
prices = yf.download(
    download_tickers,
    start=min_date.strftime("%Y-%m-%d"),
    end=max_date.strftime("%Y-%m-%d"),
    auto_adjust=True,
)["Close"]

# If only one ticker (besides SPY), yf.download returns a Series — handle both cases
if isinstance(prices, pd.Series):
    prices = prices.to_frame(name=download_tickers[0])

print(f"Price data shape: {prices.shape}, date range: {prices.index[0].date()} to {prices.index[-1].date()}")

# ── Compute returns for each filing ─────────────────────────────────────
returns_rows = []
for _, row in filing_pairs.iterrows():
    ticker = row["ticker"]
    form = row["form"]
    fdate = row["filing_date"]

    # Find filing date or next trading day in price index
    idx = prices.index.searchsorted(fdate)
    if idx >= len(prices.index):
        returns_rows.append({
            "ticker": ticker, "form": form, "filing_date": fdate,
            "ret_21d": np.nan, "ret_21d_spy": np.nan, "ret_21d_excess": np.nan,
            "status": "missing_price",
        })
        continue

    t0 = idx
    t21 = min(t0 + 21, len(prices.index) - 1)

    try:
        p0 = prices.iloc[t0][ticker]
        p21 = prices.iloc[t21][ticker]
        ret_21d = p21 / p0 - 1 if pd.notna(p0) and p0 != 0 else np.nan

        spy0 = prices.iloc[t0]["SPY"]
        spy21 = prices.iloc[t21]["SPY"]
        ret_21d_spy = spy21 / spy0 - 1 if pd.notna(spy0) and spy0 != 0 else np.nan

        ret_21d_excess = ret_21d - ret_21d_spy if pd.notna(ret_21d) and pd.notna(ret_21d_spy) else np.nan
        status = "ok" if pd.notna(ret_21d) else "missing_price"
    except (KeyError, IndexError):
        ret_21d = ret_21d_spy = ret_21d_excess = np.nan
        status = "missing_price"

    returns_rows.append({
        "ticker": ticker, "form": form, "filing_date": fdate,
        "ret_21d": ret_21d, "ret_21d_spy": ret_21d_spy,
        "ret_21d_excess": ret_21d_excess, "status": status,
    })

returns_df = pd.DataFrame(returns_rows)
returns_df.to_csv(RETURNS_CSV_PATH, index=False)
print(f"\nSaved {len(returns_df)} rows to {RETURNS_CSV_PATH}")
returns_df

## Step 4: Merge Factors with Returns

In [ ]:
# Reload from CSV to ensure consistency
returns = pd.read_csv(RETURNS_CSV_PATH, parse_dates=["filing_date"])

df = factors_df.merge(
    returns[["ticker", "filing_date", "ret_21d", "ret_21d_excess"]],
    on=["ticker", "filing_date"],
    how="left",
)

print(f"Merged shape: {df.shape}")
print(f"Factors with returns: {df['ret_21d_excess'].notna().sum()} / {len(df)}")

# Filing-level summary
filing_summary = (
    df.groupby(["ticker", "filing_date", "form"])
    .agg(
        num_factors=("key", "count"),
        avg_score=("score", "mean"),
        ret_21d_excess=("ret_21d_excess", "first"),
    )
    .reset_index()
    .sort_values("filing_date")
)
print("\nFiling-level summary:")
filing_summary

## Step 5: Factor Coverage Analysis

Identify which factors appear consistently across filings vs. sporadically.

In [ ]:
n_filings = df.groupby(["ticker", "filing_date"]).ngroups
factor_counts = df.groupby("key")["filing_date"].nunique().sort_values(ascending=False)

consistent = factor_counts[factor_counts >= 3]
sporadic = factor_counts[factor_counts <= 2]
unique_to_one = factor_counts[factor_counts == 1]

print(f"Total filings: {n_filings}")
print(f"Total unique factor keys: {len(factor_counts)}")
print(f"Consistent factors (3+ filings): {len(consistent)}")
print(f"Sporadic factors (1-2 filings): {len(sporadic)}")
print(f"Unique to one filing: {len(unique_to_one)}")

# List consistent factors with their categories
print(f"\n{'='*60}")
print(f"CONSISTENT FACTORS ({len(consistent)}):")
key_to_cat = df.drop_duplicates("key").set_index("key")["category"]
for key, count in consistent.items():
    cat = key_to_cat.get(key, "?")
    print(f"  [{cat}] {key} — {count}/{n_filings} filings")

if len(unique_to_one) > 0:
    print(f"\nFACTORS UNIQUE TO ONE FILING ({len(unique_to_one)}):")
    for key in unique_to_one.index:
        fdate = df.loc[df["key"] == key, "filing_date"].iloc[0]
        cat = key_to_cat.get(key, "?")
        print(f"  [{cat}] {key} — only in {fdate.date()}")

## Step 6: Sentiment Visualizations

Two charts:
1. **Sentiment Cohort Stacked Bar** — label distribution per filing, annotated with excess returns
2. **Category-Level Sentiment Heatmap** — average sentiment score by category and filing date

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Viz 1: Sentiment Cohort Stacked Bar ────────────────────────────────
ax1 = axes[0]

label_order = ["very_negative", "negative", "neutral", "positive", "very_positive"]
label_colors = ["#d62728", "#ff7f0e", "#999999", "#2ca02c", "#1f77b4"]

cohort = (
    df.groupby(["filing_date", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=label_order, fill_value=0)
)

# Format filing dates for x-axis labels
x_labels = [d.strftime("%Y-%m-%d") for d in cohort.index]

cohort.plot(
    kind="bar", stacked=True, color=label_colors, ax=ax1, width=0.7,
)
ax1.set_xticklabels(x_labels, rotation=45, ha="right")
ax1.set_xlabel("Filing Date")
ax1.set_ylabel("Number of Factors")
ax1.set_title("Sentiment Distribution per Filing")
ax1.legend(title="Label", fontsize=8, title_fontsize=9)

# Annotate with excess returns
filing_ret = filing_summary.set_index("filing_date")["ret_21d_excess"]
for i, fdate in enumerate(cohort.index):
    ret = filing_ret.get(fdate, np.nan)
    if pd.notna(ret):
        total_h = cohort.iloc[i].sum()
        ax1.text(
            i, total_h + 0.5, f"{ret:+.1%}",
            ha="center", va="bottom", fontsize=9, fontweight="bold",
            color="green" if ret > 0 else "red",
        )

# ── Viz 2: Category-Level Sentiment Heatmap ──────────────────────────────
ax2 = axes[1]

cat_pivot = df.pivot_table(
    values="score", index="category", columns="filing_date", aggfunc="mean",
)
cat_pivot.columns = [d.strftime("%Y-%m-%d") for d in cat_pivot.columns]

sns.heatmap(
    cat_pivot, cmap="RdYlGn", center=0, annot=True, fmt=".1f",
    linewidths=0.5, ax=ax2, cbar_kws={"label": "Avg Score"},
)
ax2.set_title("Category-Level Avg Sentiment by Filing")
ax2.set_xlabel("Filing Date")
ax2.set_ylabel("")

plt.tight_layout()
plt.show()

## Step 7: Factor Evolution & Statistical Analysis

- Factor-level evolution heatmap (consistent factors only)
- Filing-level correlation: avg sentiment vs excess return
- Factor-level information coefficient (IC)

> **N=4 filings** — all correlations are illustrative only.

In [ ]:
from scipy.stats import spearmanr

consistent_keys = factor_counts[factor_counts >= 3].index.tolist()

# ── Viz 3: Factor Evolution Heatmap ──────────────────────────────────────
evo_df = df[df["key"].isin(consistent_keys)].copy()
evo_pivot = evo_df.pivot_table(
    values="score", index="key", columns="filing_date", aggfunc="first",
)

# Sort rows by variance (most volatile at top)
row_var = evo_pivot.var(axis=1).sort_values(ascending=False)
evo_pivot = evo_pivot.loc[row_var.index]
evo_pivot.columns = [d.strftime("%Y-%m-%d") for d in evo_pivot.columns]

# Label abbreviations for annotation
score_to_abbr = {-2: "V-", -1: "N-", 0: "0", 1: "P+", 2: "V+"}
annot = evo_pivot.map(lambda x: score_to_abbr.get(int(x), "") if pd.notna(x) else "")

fig, ax = plt.subplots(figsize=(12, max(6, len(evo_pivot) * 0.4)))
sns.heatmap(
    evo_pivot, cmap="RdYlGn", center=0, annot=annot, fmt="",
    linewidths=0.5, ax=ax, cbar_kws={"label": "Score"},
)
ax.set_title(f"Factor Evolution — {len(consistent_keys)} Consistent Factors (sorted by variance)")
ax.set_xlabel("Filing Date")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

# ── Filing-level correlation ─────────────────────────────────────────────
print("="*60)
print("FILING-LEVEL CORRELATION")
print("="*60)

fs = filing_summary.dropna(subset=["ret_21d_excess"]).copy()
if len(fs) >= 3:
    corr, pval = spearmanr(fs["avg_score"], fs["ret_21d_excess"])
    print(f"Spearman(avg_score, ret_21d_excess): rho={corr:.3f}, p={pval:.3f}")
else:
    print("Not enough data points for correlation")

for _, row in fs.iterrows():
    print(f"  {row['filing_date'].date()}: avg_score={row['avg_score']:.2f}, ret_21d_excess={row['ret_21d_excess']:+.2%}")

# ── Factor-level IC ──────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("FACTOR-LEVEL INFORMATION COEFFICIENTS")
print("="*60)

ic_rows = []
for key in consistent_keys:
    fac_data = df[df["key"] == key].dropna(subset=["ret_21d_excess"]).copy()
    if len(fac_data) >= 3:
        ic, ic_p = spearmanr(fac_data["score"], fac_data["ret_21d_excess"])
        ic_rows.append({
            "key": key,
            "category": fac_data["category"].iloc[0],
            "n": len(fac_data),
            "IC": ic,
            "p_value": ic_p,
        })

ic_df = pd.DataFrame(ic_rows).sort_values("IC", key=abs, ascending=False)
print(ic_df.to_string(index=False))

# ── Scatter plot: avg sentiment vs excess return ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(fs["avg_score"], fs["ret_21d_excess"], s=100, zorder=5)
for _, row in fs.iterrows():
    ax.annotate(
        row["filing_date"].strftime("%Y-%m-%d"),
        (row["avg_score"], row["ret_21d_excess"]),
        textcoords="offset points", xytext=(8, 5), fontsize=9,
    )

# Regression line
if len(fs) >= 2:
    z = np.polyfit(fs["avg_score"], fs["ret_21d_excess"], 1)
    p = np.poly1d(z)
    x_range = np.linspace(fs["avg_score"].min() - 0.1, fs["avg_score"].max() + 0.1, 50)
    ax.plot(x_range, p(x_range), "--", color="gray", alpha=0.7, label=f"OLS fit")
    ax.legend()

ax.set_xlabel("Filing Avg Sentiment Score")
ax.set_ylabel("21-day Excess Return")
ax.set_title("Filing Sentiment vs Post-Filing Excess Return")
ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nN=4 filings — all correlations are illustrative only.")

## Step 7b: Factor Significance on Stock Movements

Quantify each factor and category's measurable impact on post-filing returns:
1. **Factor-level**: IC, mean return when positive vs negative, score variance
2. **Category-level**: IC, directional alignment with returns
3. **Composite significance ranking** combining coverage, IC magnitude, and directional consistency

In [ ]:
# ── Factor-level significance ────────────────────────────────────────
# For every factor, compute metrics relating its sentiment to returns:
# - mean_score: average sentiment score across filings
# - score_range: max - min score (volatility of sentiment)
# - coverage: number of filings the factor appears in
# - mean_ret_positive: mean excess return when factor sentiment > 0
# - mean_ret_negative: mean excess return when factor sentiment < 0
# - directional_hit_rate: fraction of filings where sign(score) == sign(ret)
# - IC: Spearman correlation (only for factors with 3+ observations)

sig_rows = []
for key, grp in df.groupby("key"):
    grp_valid = grp.dropna(subset=["ret_21d_excess"])
    if len(grp_valid) == 0:
        continue

    scores = grp_valid["score"]
    rets = grp_valid["ret_21d_excess"]

    pos_mask = scores > 0
    neg_mask = scores < 0

    # Directional hit rate: how often does score sign match return sign?
    nonzero = scores != 0
    if nonzero.sum() > 0:
        hits = ((scores[nonzero] > 0) == (rets[nonzero] > 0)).mean()
    else:
        hits = np.nan

    # IC only if enough data
    ic_val = np.nan
    ic_p = np.nan
    if len(grp_valid) >= 3 and scores.nunique() > 1:
        ic_val, ic_p = spearmanr(scores, rets)

    sig_rows.append({
        "key": key,
        "category": grp_valid["category"].iloc[0],
        "coverage": len(grp_valid),
        "mean_score": scores.mean(),
        "score_range": scores.max() - scores.min(),
        "mean_ret_positive": rets[pos_mask].mean() if pos_mask.sum() > 0 else np.nan,
        "mean_ret_negative": rets[neg_mask].mean() if neg_mask.sum() > 0 else np.nan,
        "directional_hit_rate": hits,
        "IC": ic_val,
        "IC_pvalue": ic_p,
    })

significance_df = pd.DataFrame(sig_rows)

# Composite significance score (higher = more significant):
#   |IC| * coverage_weight * (1 if directional_hit_rate > 0.5 else 0.5)
# For factors without IC, fall back to |mean_score| * coverage / n_filings
significance_df["significance"] = significance_df.apply(
    lambda r: (
        abs(r["IC"]) * (r["coverage"] / n_filings) * (1.0 if r["directional_hit_rate"] > 0.5 else 0.5)
        if pd.notna(r["IC"])
        else abs(r["mean_score"]) * r["coverage"] / (n_filings * 5)  # fallback for low-coverage factors
    ),
    axis=1,
)

significance_df = significance_df.sort_values("significance", ascending=False).reset_index(drop=True)

print("="*70)
print("FACTOR SIGNIFICANCE ON STOCK MOVEMENTS")
print("="*70)
display_cols = ["key", "category", "coverage", "mean_score", "score_range",
                "directional_hit_rate", "IC", "significance"]
print(significance_df[display_cols].to_string(index=False, float_format="%.3f"))

# ── Category-level significance ──────────────────────────────────────
print(f"\n{'='*70}")
print("CATEGORY-LEVEL SIGNIFICANCE")
print("="*70)

cat_sig_rows = []
for cat, cgrp in df.groupby("category"):
    # One observation per (category, filing_date): mean score
    cat_filing = cgrp.groupby("filing_date").agg(
        avg_score=("score", "mean"),
        ret_21d_excess=("ret_21d_excess", "first"),
    ).dropna()
    if len(cat_filing) < 3:
        cat_sig_rows.append({"category": cat, "n_filings": len(cat_filing),
                             "IC": np.nan, "IC_pvalue": np.nan,
                             "mean_score": cat_filing["avg_score"].mean()})
        continue
    ic_val, ic_p = spearmanr(cat_filing["avg_score"], cat_filing["ret_21d_excess"])
    cat_sig_rows.append({"category": cat, "n_filings": len(cat_filing),
                         "IC": ic_val, "IC_pvalue": ic_p,
                         "mean_score": cat_filing["avg_score"].mean()})

cat_sig_df = pd.DataFrame(cat_sig_rows).sort_values("IC", key=abs, ascending=False)
print(cat_sig_df.to_string(index=False, float_format="%.3f"))

# ── Visualization: top factors by significance ───────────────────────
top_n = min(20, len(significance_df))
top = significance_df.head(top_n).copy()

fig, ax = plt.subplots(figsize=(10, max(4, top_n * 0.35)))
colors = ["#2ca02c" if s > 0 else "#d62728" if s < 0 else "#999999" for s in top["mean_score"]]
bars = ax.barh(range(top_n), top["significance"], color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels([f"{r['key']} ({r['category']})" for _, r in top.iterrows()], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Significance Score")
ax.set_title("Factor Significance on Stock Movements (ranked)")

# Annotate with IC and hit rate
for i, (_, r) in enumerate(top.iterrows()):
    ic_str = f"IC={r['IC']:.2f}" if pd.notna(r["IC"]) else "IC=n/a"
    hr_str = f"hit={r['directional_hit_rate']:.0%}" if pd.notna(r["directional_hit_rate"]) else ""
    ax.text(r["significance"] + 0.005, i, f"{ic_str} {hr_str}", va="center", fontsize=8)

# Legend for color
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2ca02c", label="Net positive sentiment"),
    Patch(facecolor="#d62728", label="Net negative sentiment"),
    Patch(facecolor="#999999", label="Neutral sentiment"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nN={n_filings} filings — significance scores are illustrative. "
      f"Scaling to all tickers will strengthen statistical power.")

## Step 8: LLM Client & Reasoning Prompts

Reuses the same LLM client pattern as task_1_3. Defines two reasoning prompts:
1. **Pairwise**: Compare two consecutive filings to identify changed/persistent/emerging factors
2. **Synthesis**: Aggregate pairwise findings with returns to identify predictive patterns

> **Requires vLLM on ACCRE.** Steps 8-9 will skip gracefully if the server is unreachable.

In [ ]:
# ── LLM Client (adapted for DeepSeek-R1 reasoning model) ────────────────
import re

_semaphore = threading.BoundedSemaphore(MAX_IN_FLIGHT_LLM)

# Reasoning models need more tokens: thinking + JSON output
REASONING_MAX_TOKENS = 16384


def call_llm(
    client: "OpenAI",
    system_prompt: str,
    user_prompt: str,
    temperature: float = TEMPERATURE,
    max_tokens: int = REASONING_MAX_TOKENS,
) -> Optional[str]:
    """Call the LLM with retry logic and concurrency control."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with _semaphore:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens,
                    timeout=REQUEST_TIMEOUT,
                )
            return response.choices[0].message.content
        except Exception as e:
            delay = RETRY_BASE_DELAY * (2 ** (attempt - 1))
            log.warning(f"  LLM call attempt {attempt}/{MAX_RETRIES} failed: {e}. Retrying in {delay}s...")
            if attempt < MAX_RETRIES:
                time.sleep(delay)
            else:
                log.error(f"  LLM call failed after {MAX_RETRIES} attempts: {e}")
                return None


def parse_json_response(content: str) -> Optional[Any]:
    """Parse JSON from LLM response, handling <think> blocks and markdown code blocks."""
    content = content.strip()

    # Strip DeepSeek-R1 <think>...</think> reasoning blocks (closed)
    content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()

    # Strip unclosed <think> blocks (truncated response — no </think>)
    if "<think>" in content:
        content = re.sub(r"<think>.*", "", content, flags=re.DOTALL).strip()

    if content.startswith("```"):
        content = content.split("\n", 1)[1] if "\n" in content else content[3:]
        if content.endswith("```"):
            content = content[:-3]
        content = content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass
    start_obj = content.find("{")
    start_arr = content.find("[")
    if start_obj == -1 and start_arr == -1:
        log.warning(f"  No JSON found in response ({len(content)} chars). First 300: {content[:300]}")
        return None
    if start_arr != -1 and (start_obj == -1 or start_arr < start_obj):
        start = start_arr
        end = content.rfind("]")
    else:
        start = start_obj
        end = content.rfind("}")
    if end == -1 or end <= start:
        log.warning(f"  Malformed JSON in response. First 300: {content[:300]}")
        return None
    try:
        return json.loads(content[start:end + 1])
    except json.JSONDecodeError as e:
        log.warning(f"  JSON parse error: {e}. First 300: {content[start:start+300]}")
        return None


def call_llm_json(
    client: "OpenAI",
    system_prompt: str,
    user_prompt: str,
    temperature: float = TEMPERATURE,
    max_tokens: int = REASONING_MAX_TOKENS,
) -> Optional[Any]:
    """Call the LLM and parse the response as JSON."""
    content = call_llm(client, system_prompt, user_prompt, temperature, max_tokens)
    if content is None:
        return None
    return parse_json_response(content)


# ── Client creation + health check ───────────────────────────────────────
llm_available = False
if OpenAI is not None:
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
    try:
        models = client.models.list()
        available = [m.id for m in models.data]
        print(f"vLLM is reachable. Available models: {available}")
        if MODEL not in available:
            print(f"WARNING: {MODEL} not in available models!")
        llm_available = True
    except Exception as e:
        print(f"WARNING: vLLM not reachable ({e}).")
        print("Steps 8-9 (LLM reasoning) will be skipped.")
        print("Steps 1-7 (data + statistical analysis) still work without GPU.")
else:
    print("openai package not installed. Steps 8-9 will be skipped.")

# ── Reasoning Prompts ────────────────────────────────────────────────────
# DeepSeek-R1 reasoning models often ignore system prompt formatting rules.
# We use a minimal system prompt and embed the JSON schema in the user prompt.

REASONING_SYSTEM_PROMPT = "You are a financial analyst. Always respond with valid JSON only, no other text outside the JSON object."

PAIRWISE_JSON_SCHEMA = """\
You MUST respond with ONLY a JSON object in this exact structure (no other text):
```json
{
  "changed_factors": [
    {"key": "<factor_key>", "from_label": "<prev_label>", "to_label": "<curr_label>", "interpretation": "<1-2 sentences>"}
  ],
  "persistent_themes": [
    {"key": "<factor_key>", "label": "<consistent_label>", "interpretation": "<1-2 sentences>"}
  ],
  "emerged_or_disappeared": [
    {"key": "<factor_key>", "direction": "emerged or disappeared", "interpretation": "<1-2 sentences>"}
  ],
  "overall_narrative": "<2-3 sentences>"
}
```"""

SYNTHESIS_JSON_SCHEMA = """\
You MUST respond with ONLY a JSON object in this exact structure (no other text):
```json
{
  "key_patterns": [
    {
      "pattern": "<describe the pattern>",
      "supporting_factors": ["<factor_key_1>", "<factor_key_2>"],
      "direction": "positive or negative or mixed",
      "strength": "strong or moderate or weak",
      "explanation": "<2-3 sentences>"
    }
  ],
  "most_predictive_factors": [
    {"key": "<factor_key>", "rationale": "<1-2 sentences>", "confidence": "high or medium or low"}
  ],
  "overall_narrative": "<3-5 sentences>"
}
```"""

print(f"Reasoning prompts loaded. Max tokens: {REASONING_MAX_TOKENS}")

## Step 9: Run LLM Reasoning

1. **Pairwise**: Compare each consecutive pair of filings (3 LLM calls for 4 filings)
2. **Synthesis**: Aggregate pairwise results with returns data (1 LLM call)

Results saved to `output/reasoning_output.json`.

In [ ]:
if not llm_available:
    print("Skipping LLM reasoning — vLLM server not available.")
    print("Run this notebook on ACCRE with vLLM to execute Steps 8-9.")
else:
    # Sort filings by date
    filings_sorted = sorted(df["filing_date"].unique())
    print(f"Filings in order: {[str(d.date()) for d in filings_sorted]}")
    print(f"Consecutive pairs: {len(filings_sorted) - 1}")

    def _format_factors_for_prompt(filing_df: pd.DataFrame) -> str:
        """Format a filing's factors as text for the LLM prompt."""
        lines = []
        for _, row in filing_df.iterrows():
            lines.append(
                f"- [{row['category']}] {row['key']}: {row['label']}\n"
                f"  Summary: {row['summary']}"
            )
        return "\n".join(lines)

    # ── Pairwise comparisons ──────────────────────────────────────────────
    pairwise_results = []

    for i in range(len(filings_sorted) - 1):
        date_a = filings_sorted[i]
        date_b = filings_sorted[i + 1]
        print(f"\n{'='*60}")
        print(f"Pairwise {i+1}/{len(filings_sorted)-1}: {date_a.date()} → {date_b.date()}")

        df_a = df[df["filing_date"] == date_a]
        df_b = df[df["filing_date"] == date_b]

        ret_a = df_a["ret_21d_excess"].iloc[0] if df_a["ret_21d_excess"].notna().any() else None
        ret_b = df_b["ret_21d_excess"].iloc[0] if df_b["ret_21d_excess"].notna().any() else None

        user_prompt = (
            f"Compare these two consecutive SEC filings and identify changes in thematic factors.\n\n"
            f"FILING A — {date_a.date()} (21d excess return: {f'{ret_a:+.2%}' if ret_a is not None else 'N/A'}):\n"
            f"{_format_factors_for_prompt(df_a)}\n\n"
            f"FILING B — {date_b.date()} (21d excess return: {f'{ret_b:+.2%}' if ret_b is not None else 'N/A'}):\n"
            f"{_format_factors_for_prompt(df_b)}\n\n"
            f"{PAIRWISE_JSON_SCHEMA}"
        )

        result = call_llm_json(client, REASONING_SYSTEM_PROMPT, user_prompt)
        if result:
            pairwise_results.append({
                "filing_a": str(date_a.date()),
                "filing_b": str(date_b.date()),
                "ret_a": ret_a,
                "ret_b": ret_b,
                "result": result,
            })
            narrative = result.get("overall_narrative", "(no narrative)")
            print(f"  Narrative: {narrative}")
            changed = result.get("changed_factors", [])
            print(f"  Changed factors: {len(changed)}")
        else:
            pairwise_results.append({
                "filing_a": str(date_a.date()),
                "filing_b": str(date_b.date()),
                "ret_a": ret_a,
                "ret_b": ret_b,
                "result": None,
            })
            print("  WARNING: LLM call failed")

    # ── Synthesis ─────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("SYNTHESIS")

    # Build category-level averages per filing
    cat_scores = df.pivot_table(
        values="score", index="category", columns="filing_date", aggfunc="mean",
    )
    cat_scores.columns = [str(d.date()) for d in cat_scores.columns]
    cat_text = cat_scores.to_string()

    # Build returns summary
    ret_text = "\n".join(
        f"  {row['filing_date'].date()}: ret_21d_excess={row['ret_21d_excess']:+.2%}"
        for _, row in filing_summary.dropna(subset=["ret_21d_excess"]).iterrows()
    )

    # Build pairwise narratives
    pair_text = "\n\n".join(
        f"Pair {i+1} ({p['filing_a']} → {p['filing_b']}):\n"
        f"{p['result'].get('overall_narrative', '(failed)')}" if p["result"] else f"Pair {i+1}: (failed)"
        for i, p in enumerate(pairwise_results)
    )

    synthesis_prompt = (
        f"Synthesize this multi-year filing analysis. Identify which factors connect to stock returns.\n\n"
        f"PAIRWISE SUMMARIES:\n{pair_text}\n\n"
        f"RETURNS:\n{ret_text}\n\n"
        f"CATEGORY-LEVEL AVG SENTIMENT SCORES:\n{cat_text}\n\n"
        f"{SYNTHESIS_JSON_SCHEMA}"
    )

    synthesis_result = call_llm_json(client, REASONING_SYSTEM_PROMPT, synthesis_prompt)
    if synthesis_result:
        print(f"  Narrative: {synthesis_result.get('overall_narrative', '(no narrative)')}")
        print(f"  Key patterns: {len(synthesis_result.get('key_patterns', []))}")
        print(f"  Most predictive factors: {len(synthesis_result.get('most_predictive_factors', []))}")
    else:
        print("  WARNING: Synthesis LLM call failed")

    # ── Save results ─────────────────────────────────────────────────────
    reasoning_output = {
        "ticker": TICKERS[0],
        "model": MODEL,
        "num_filings": len(filings_sorted),
        "pairwise": pairwise_results,
        "synthesis": synthesis_result,
    }

    with open(REASONING_OUTPUT_PATH, "w") as f:
        json.dump(reasoning_output, f, indent=2, default=str)
    print(f"\nSaved reasoning output to {REASONING_OUTPUT_PATH}")

## Step 10: Reasoning Results

Display LLM reasoning results: pairwise narratives, synthesis patterns, and cross-reference with statistical IC.

In [ ]:
# Load reasoning output (works whether Step 9 just ran or from a previous session)
if REASONING_OUTPUT_PATH.exists():
    with open(REASONING_OUTPUT_PATH) as f:
        reasoning = json.load(f)

    # ── Pairwise results ──────────────────────────────────────────────────
    for i, pair in enumerate(reasoning.get("pairwise", [])):
        result = pair.get("result")
        if not result:
            continue
        print(f"{'='*60}")
        print(f"Pair {i+1}: {pair['filing_a']} → {pair['filing_b']}")
        print(f"  Returns: {pair.get('filing_a', '?')}={pair.get('ret_a', '?')}, "
              f"{pair.get('filing_b', '?')}={pair.get('ret_b', '?')}")
        print(f"\n  Narrative: {result.get('overall_narrative', '')}")

        changed = result.get("changed_factors", [])
        if changed:
            print(f"\n  Top changed factors:")
            for cf in changed[:3]:
                print(f"    {cf.get('key', '?')}: {cf.get('from_label', '?')} → {cf.get('to_label', '?')}")
                print(f"      {cf.get('interpretation', '')}")
        print()

    # ── Synthesis results ─────────────────────────────────────────────────
    synthesis = reasoning.get("synthesis")
    if synthesis:
        print(f"{'='*60}")
        print("SYNTHESIS")
        print(f"{'='*60}")

        patterns = synthesis.get("key_patterns", [])
        if patterns:
            print(f"\nKey Patterns ({len(patterns)}):")
            for j, pat in enumerate(patterns, 1):
                print(f"  {j}. {pat.get('pattern', '?')}")
                print(f"     Direction: {pat.get('direction', '?')}, Strength: {pat.get('strength', '?')}")
                print(f"     Factors: {', '.join(pat.get('supporting_factors', []))}")
                print(f"     {pat.get('explanation', '')}")

        predictive = synthesis.get("most_predictive_factors", [])
        if predictive:
            print(f"\nMost Predictive Factors ({len(predictive)}):")
            # Build IC lookup for cross-reference
            ic_lookup = {}
            if 'ic_df' in dir() and len(ic_df) > 0:
                ic_lookup = dict(zip(ic_df["key"], ic_df["IC"]))

            for j, pf in enumerate(predictive, 1):
                key = pf.get("key", "?")
                stat_ic = ic_lookup.get(key)
                ic_str = f", Statistical IC={stat_ic:.3f}" if stat_ic is not None else ""
                print(f"  {j}. {key} (confidence: {pf.get('confidence', '?')}{ic_str})")
                print(f"     {pf.get('rationale', '')}")

        print(f"\nOverall Narrative:")
        print(f"  {synthesis.get('overall_narrative', '(none)')}")
    else:
        print("No synthesis results available.")
else:
    print(f"No reasoning output found at {REASONING_OUTPUT_PATH}")
    print("Run Steps 8-9 on ACCRE with vLLM to generate reasoning results.")

## Step 11: Output Summary

In [ ]:
print("="*60)
print("OUTPUT SUMMARY")
print("="*60)

output_files = [
    ("Filing returns", RETURNS_CSV_PATH),
    ("Reasoning output", REASONING_OUTPUT_PATH),
]
for label, path in output_files:
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  {label}: {path.name} ({size_kb:.1f} KB)")
    else:
        print(f"  {label}: {path.name} (not yet created)")

print(f"\nKey Stats:")
print(f"  Tickers analyzed: {TICKERS}")
print(f"  Filings analyzed: {factors_df.groupby(['ticker', 'filing_date']).ngroups}")
print(f"  Total factors: {len(factors_df)}")
print(f"  Consistent factors (3+ filings): {len(consistent_keys)}")

if RETURNS_CSV_PATH.exists():
    ret = pd.read_csv(RETURNS_CSV_PATH)
    ok_count = (ret["status"] == "ok").sum() if "status" in ret.columns else len(ret)
    print(f"  Filings with valid returns: {ok_count}/{len(ret)}")

if REASONING_OUTPUT_PATH.exists():
    with open(REASONING_OUTPUT_PATH) as f:
        r = json.load(f)
    n_pairs = len([p for p in r.get("pairwise", []) if p.get("result")])
    has_synth = r.get("synthesis") is not None
    print(f"  Pairwise comparisons: {n_pairs}")
    print(f"  Synthesis: {'yes' if has_synth else 'no'}")

print(f"\nTo run all 87 tickers: comment out the TICKERS filter in Step 1.")